In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from global_parameters import Assumptions

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

from AileronSizing.aileron_sizing import size_ailerons

# Creating combinations of planforms

In [2]:
span_max = 3.5 #NOTE changeable to meet the requirements

lists_to_recombine = dict()

lists_to_recombine['aspect_ratio'] = [5., 10., 17., 27.]
lists_to_recombine['taper'] = [1.]
lists_to_recombine['thickness_to_chord'] = [.08, .12, .18] 
lists_to_recombine['sweep'] = [0., 20.] #NOTE constrained by tail sizing
lists_to_recombine['cl_alpha'] = [2*np.pi]
lists_to_recombine['cl_max'] = [1.]
lists_to_recombine['cm_ac'] = [-.15, 0.05] 
lists_to_recombine['cl_0'] = [0.]
lists_to_recombine['pf_type'] = ['tail', 'canard']

In [8]:
ltr_keys = lists_to_recombine.keys()
ltr_values = lists_to_recombine.values()

planforms_raw = list(itt.product(*ltr_values))
print(planforms_raw)

planform_params = list()
for planform_raw in planforms_raw:
    planform_param = dict()
    for i, key in enumerate(ltr_keys):
        planform_param[key] = planform_raw[i]
    planform_params.append(planform_param)

assert len(planform_params) == 96, len(planform_params)

[(5.0, 1.0, 0.08, 0.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.08, 0.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.08, 0.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.08, 0.0, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.08, 20.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.08, 20.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.08, 20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.08, 20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, 0.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.12, 0.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.12, 0.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.12, 0.0, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, 20.0, 6.283185307179586, 1.0, -0.15, 0.0, 'tail'), (5.0, 1.0, 0.12, 20.0, 6.283185307179586, 1.0, -0.15, 0.0, 'canard'), (5.0, 1.0, 0.12, 20.0, 6.283185307179586, 1.0, 

In [4]:
wing_area = span_max**2/max(lists_to_recombine['aspect_ratio'])

planforms:list[tuple[Planform, str, bool]] = list()

for planform_param in planform_params:
    span = np.sqrt(wing_area * planform_param['aspect_ratio'])

    planforms.append((Planform(
        aspect_ratio=planform_param['aspect_ratio'],
        taper=planform_param['taper'],
        sweep_quarter_deg=planform_param['sweep'],
        thickness_to_chord=planform_param['thickness_to_chord'],
        cm_quarter_chord=planform_param['cm_ac'],
        cl0=planform_param['cl_0'],
        clmax=planform_param['cl_max'],
        flap=False, #NOTE for now
        airfoil_lift_slope=planform_param['cl_alpha'],
        wetted_surface_ratio=1.07,
        interference_factor=1.,
        span=span
    ), planform_param["pf_type"]))

In [5]:
print(planforms)

[(<Aircraft.Planform.Planform object at 0x000002BDB5A04E60>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDE4385940>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDE4387110>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5A96060>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDB5A97EF0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5A97F50>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDB529B3B0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC86E0>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC8050>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC8530>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC8440>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC8320>, 'canard'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC8770>, 'tail'), (<Aircraft.Planform.Planform object at 0x000002BDB5AC87D0>, 'canard'), (<Aircraft.Planform

# Caching Planform properties

## CD0

In [6]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

for planform in planforms:
    planform[0].add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    planform[0].add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
    #NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    planform[0].add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    planform[0].add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

## Planform Mass

In [ ]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

for planform in planforms:
    size_planform(planform=planform[0], thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=0.315, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

Stresses 26202968.353814386, 80454660.26233284, 0.0004
Stresses 26202968.353814386, 80454660.26233284, 0.0004
Stresses 26944504.283644263, 80454660.26233284, 0.0004
Stresses 26944504.283644263, 80454660.26233284, 0.0004
Stresses 26202968.353814386, 80454660.26233284, 0.0004
Stresses 26202968.353814386, 80454660.26233284, 0.0004
Stresses 26944504.283644263, 80454660.26233284, 0.0004
Stresses 26944504.283644263, 80454660.26233284, 0.0004
Stresses 17489629.188129183, 52348608.70386616, 0.0004
Stresses 17489629.188129183, 52348608.70386616, 0.0004
Stresses 17983986.474682435, 52348608.70386616, 0.0004
Stresses 17983986.474682435, 52348608.70386616, 0.0004
Stresses 17489629.188129183, 52348608.70386616, 0.0004
Stresses 17489629.188129183, 52348608.70386616, 0.0004
Stresses 17983986.474682435, 52348608.70386616, 0.0004
Stresses 17983986.474682435, 52348608.70386616, 0.0004
Stresses 11688250.568818517, 33972271.73718672, 0.0004
Stresses 11688250.568818517, 33972271.73718672, 0.0004
Stresses 1

# Saving the planforms

In [10]:
with open("pickles/planform_pickle_official.pcl", "w+b") as f:
    pickle.dump(planforms, f)

### Recovery to see if pickled correctly

In [11]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered = pickle.load(f)

In [13]:
sample:Planform = plaforms_recovered[5][0]
print(f"AR: {sample.aspect_ratio}")
print(f"cr: {sample.c_root}")
print(f"CD0 takeoff: {sample.CD0_cache["takeoff"]}")

pf_masses = [pf[0].mass_cache for pf in plaforms_recovered]
print(max(pf_masses), min(pf_masses))
print(sample.span)
print(span_max)

AR: 5.0
cr: 0.3012320380383546
CD0 takeoff: 0.003996529764628093
4.435500752111364 2.9149367006944886
1.5061601901917732
3.5


Checking if the ailerons fit

In [12]:
for pf in plaforms_recovered:
    wing = pf[0]
    print(size_ailerons(np.deg2rad(20.), Vmin=38., Vmax=230., chords_ratio=0.3, n_sections=100, planform=wing, roll_rate_max=np.deg2rad(180), roll_rate_min=np.deg2rad(60.), y_fus=0.33/2))

[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.2535881952873904, 0.2689571768199595), (0.5917057890039109, 0.6301282428353336)]
[(0.2535881952873904, 0.2689571768199595), (0.5917057890039109, 0.6301282428353336)]
[(0.2535881952873904, 0.2689571768199595), (0.5917057890039109, 0.6301282428353336)]
[(0.2535881952873904, 0.2689571768199595), (0.5917057890039109, 0.6301282428353336)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.7223421320307484, 0.7530800950958866)]
[(0.24590370452110583, 0.26127268605367493), (0.722